In [74]:
import pandas as pd

In [75]:
catalogo=pd.read_csv("../data/catalog/catalogo_fontes.csv")

In [76]:
arquivos=dict(
    zip(
        catalogo["fonte"],
        catalogo["arquivo"]
    )
)


In [204]:
censo=pd.read_csv(f"{arquivos['CENSO']}", encoding="latin1", sep=";", low_memory=False)

censo=censo[censo["IN_PROFISSIONALIZANTE"]==1]
colunas_censo=[
    "CO_ENTIDADE",
    "NO_REGIAO",
    "SG_UF",
    "TP_LOCALIZACAO",
    "IN_INTERNET_ALUNOS",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_BANHEIRO",
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_INEXISTENTE",
    "IN_BANHEIRO_PNE",
    "IN_ACESSIBILIDADE_CORRIMAO",
    "IN_ACESSIBILIDADE_ELEVADOR",
    "IN_ACESSIBILIDADE_VAO_LIVRE",
    "IN_ACESSIBILIDADE_RAMPAS",
    "IN_ACESSIBILIDADE_SINAL_SONORO",
    "IN_ACESSIBILIDADE_SINAL_TATIL",
    "IN_ACESSIBILIDADE_SINAL_VISUAL"
]

censo = censo[colunas_censo]

censo["TP_LOCALIZACAO"] = censo["TP_LOCALIZACAO"].map({1:'Urbana', 2:'Rural'})

indicadores_binarios=[
    "IN_INTERNET_ALUNOS",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_BANHEIRO",
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_INEXISTENTE",
    "IN_ESGOTO_INEXISTENTE",
    "IN_BANHEIRO_PNE",
    "IN_ACESSIBILIDADE_CORRIMAO",
    "IN_ACESSIBILIDADE_ELEVADOR",
    "IN_ACESSIBILIDADE_VAO_LIVRE",
    "IN_ACESSIBILIDADE_RAMPAS",
    "IN_ACESSIBILIDADE_SINAL_SONORO",
    "IN_ACESSIBILIDADE_SINAL_TATIL",
    "IN_ACESSIBILIDADE_SINAL_VISUAL"
]

for coluna in indicadores_binarios:
    censo[coluna] = pd.to_numeric(censo[coluna], errors='coerce').fillna(0).astype(int)



In [205]:
rendimento_alunos=pd.read_excel(f"{arquivos['RENDIMENTO']}", skiprows=5)
rendimento_alunos=rendimento_alunos.dropna(how='all')

In [206]:
rend_colunas=[
    "Código da Escola",
    "Nome da Escola",
    "Dependência Administrativa",
    "Unnamed: 21",
    "Unnamed: 39",
    "Unnamed: 57"
]

rendimento_alunos=rendimento_alunos[rend_colunas]

rendimento_alunos=rendimento_alunos.iloc[3:]

rendimento_alunos.rename(columns={"Unnamed: 21":"Aprovação", "Unnamed: 39":"Reprovação", "Unnamed: 57":"Abandono"}, inplace=True)

codigo=censo["CO_ENTIDADE"]
rendimento_alunos=rendimento_alunos[rendimento_alunos["Código da Escola"].isin(codigo)]


In [207]:
esc_ren=pd.merge(rendimento_alunos, censo, right_on= "CO_ENTIDADE", left_on= "Código da Escola", how= "inner")
esc_ren=esc_ren.drop(columns= ["CO_ENTIDADE"])

In [208]:
esc_ren["IN_ENERGIA_INEXISTENTE"] = esc_ren["IN_ENERGIA_INEXISTENTE"].replace({0: 1, 1: 0})
esc_ren["IN_ESGOTO_INEXISTENTE"] = esc_ren["IN_ESGOTO_INEXISTENTE"].replace({0: 1, 1: 0})
    
esc_ren.rename(columns={"IN_ENERGIA_INEXISTENTE":"Energia", "IN_ESGOTO_INEXISTENTE":"Esgoto"}, inplace=True)

In [209]:
mapa_descricoes = {
    "Código da Escola": "Código único de identificação da escola (Inep/Censo Escolar).",
    "Nome da Escola": "Nome da instituição de ensino",
    "Dependência Administrativa": "Instância administrativa à qual a escola está subordinada, na esfera federal, estadual, municipal ou privada",
    "Aprovação": "Taxa de aprovação total de alunos do ensino médio de escolas que oferecem educação profissional",
    "Reprovação": "Taxa de reprovação total de alunos do ensino médio de escolas que oferecem educação profissional",
    "Abandono": "Taxa de abandono total dos estudos de alunos do ensino médio de escolas que oferecem educação profissional",
    "NO_REGIAO": "Região do Brasil na qual a escola se localiza",
    "SG_UF": "União federativa na qual a escola se localiza",
    "TP_LOCALIZACAO": "Tipo de localização na qual a escola se encontra(rural ou urbana)",
    "IN_INTERNET_ALUNOS": "Indicador de internet para alunos: 1 para Existente, 0 para Inexistente.",
    "IN_INTERNET_APRENDIZAGEM": "Indicador de internet para uso no processo de aprendizagem: 1 para Existente, 0 para Inexistente.",
    "IN_LABORATORIO_INFORMATICA": "Presença de laboratório de informática: 1 para Existente, 0 para Inexistente",
    "IN_BIBLIOTECA":"Presença de biblioteca: 1 para Existente, 0 para Inexistente",
    "IN_BANHEIRO": "Presença de banheiro: 1 para Existente, 0 para Inexistente",
    "IN_AGUA_POTAVEL": "Disponibilidade de água potável: 1 para Existente, 0 para Inexistente", 
    "Energia": "Indicador de energia elétrica: 1 para Existente, 0 para Inexistente.",
    "Esgoto": "Indicador de esgoto sanitário: 1 para Existente, 0 para Inexistente.",
    "IN_BANHEIRO_PNE": "Presença de banheiros adaptados para pessoas com deficiência física e/ou motora: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_CORRIMAO": "Presença de corrimão para o auxílio na locomoção de pessoas com deficiência física e/ou motora: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_ELEVADOR": "Presença de elevadores para o auxílio na locomoção de pessoas com deficiência: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_VAO_LIVRE": "Presença de vão livre para facilitar na locomoção de pessoas com deficiência: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_RAMPAS": "Presença de rampa para o auxílio na locomoção de pessoas com deficiência física e/ou motora: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_SINAL_SONORO": "Recurso sonoro de acessibilidade: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_SINAL_TATIL":  "Recurso tatil de acessibilidade: 1 para Existente, 0 para Inexistente.",
    "IN_ACESSIBILIDADE_SINAL_VISUAL": "Recurso visual de acessibilidade: 1 para Existente, 0 para Inexistente."
    }

def obter_descricao_automatica(nome_coluna):
   
    if nome_coluna in mapa_descricoes:
        return mapa_descricoes[nome_coluna]

 
    elif nome_coluna.startswith("IN_"):
        termo_limpo = nome_coluna.replace("IN_", "").replace("_", " ").title()
        return f"Variável indicadora binária (0 ou 1) para: {termo_limpo}."



colunas_atuais = esc_ren.columns
tipos_dados = esc_ren.dtypes


informacoes_dicionario = {
    coluna: [str(tipo), obter_descricao_automatica(coluna)]
    for coluna, tipo in zip(colunas_atuais, tipos_dados)
}


df_dicionario = pd.DataFrame.from_dict(
    informacoes_dicionario,
    orient="index",
    columns=["Tipo_de_Dado", "Descricao"],
)

# Define o nome da coluna que guardará as chaves originais
df_dicionario.index.name = "Nome_da_Coluna"


In [210]:
df_dicionario.to_csv("../data/catalog/dicionario_dados.csv", sep=";", encoding="utf-8")
esc_ren.to_csv("../data/catalog/dados_merge.csv", sep=";", encoding="utf-8")